In [ ]:
import xarray as xr
import rioxarray
import pandas as pd

from const import PROJECTION

Read in each intermediate dataset

In [ ]:
all_files = [
    "../data_working/ads_damage.zarr/",
    "../data_working/topo.zarr/",
    "../data_working/terraclimate.zarr/",
    "../data_working/treemap2016_hostba_hydro.zarr/"
]

all_ds = [
    xr.open_zarr(f) for f in all_files
]

Cleanup

In [ ]:
all_ds[0] = all_ds[0].__xarray_dataarray_variable__.to_dataset(dim="var")

In [ ]:
all_ds[1] = all_ds[1].__xarray_dataarray_variable__.to_dataset(dim="var").drop_vars("band")

In [ ]:
all_ds[2] = all_ds[2].drop_vars("spatial_ref")

In [ ]:
all_ds[3] = all_ds[3].band_1.to_dataset(dim="var")

Combine

In [ ]:
merged = xr.combine_by_coords(all_ds, combine_attrs="drop")\
    .compute()\
    .rio.write_crs(PROJECTION)

Generate target variables. This is mortality shifted backward one year.

In [ ]:
mort_vars = list(filter(
    lambda x: x.endswith("mort"),
    list(merged.variables.keys())
))

target_vars = merged[mort_vars].shift(time=-1)

# Rename variables and merge
var_rename_dict = {
    x:x.replace("mort", "target")
    for x in mort_vars
}

target_vars_rename = target_vars.rename(**var_rename_dict)

westmort = xr.combine_by_coords([merged, target_vars_rename])\
    .assign_coords(time=pd.DatetimeIndex(merged.time))

In [ ]:
assert (
    westmort["doug_fb_target"].sel(time="2004-01-01").fillna(-1) == 
    westmort["doug_fb_mort"].sel(time="2005-01-01").fillna(-1)
).all()

In [ ]:
# Save
westmort.to_zarr("../data_working/westmort.zarr")